# 🧠 Getting Started with Agents (Local LLM + LangGraph)

This notebook teaches you **agent fundamentals** and walks you through building your **first local AI agent** using:

- Ollama (local LLM)
- Qwen3-1.7B
- OpenAI-compatible client
- LangGraph (agent framework)

---


In [ ]:
%pip install openai langgraph

## 🧠 What is an Agent?

An **AI Agent** consists of:

1. **LLM (Brain)** – makes decisions
2. **Tools (Actions)** – functions it can call
3. **Memory (State)** – retains conversation
4. **Control Loop** – decides next step

### Agent Loop
```
User Input → LLM → (Tool or Answer) → Repeat → Final Output
```


## 🔧 Main Components of an Agent

### 1. Model
- Local LLM (Ollama)

### 2. Tools
- Functions that perform actions

### 3. Memory
- Stores messages/state

### 4. Orchestration
- Loop handled by LangGraph


## 🤖 Types of Agents

1. **ReAct Agent** – Think + Act loop
2. **Tool-Calling Agent** – Uses structured tools
3. **Planner-Executor Agent** – Plans then executes
4. **Multi-Agent System** – Multiple collaborating agents


## 🧩 Agent vs Pipeline

### Pipeline
- Fixed steps
- Deterministic

### Agent
- Dynamic decisions
- Adaptive behavior
- Uses tools when needed


## 👥 Agent Roles

- **Planner** → decides strategy
- **Executor** → performs tasks
- **Tool Agent** → interacts with tools


## 🧠 Structured Reasoning

### Chain of Thought (CoT)
- Step-by-step reasoning

### Tree of Thought (ToT)
- Multiple reasoning paths
- Choose best path

**Practical Tip:**
Use CoT for simple tasks, ToT for complex problems.


## 🧩 Build Your First Agent

In [ ]:
from openai import OpenAI
from langgraph.graph import StateGraph, END
import json

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

class AgentState(dict):
    messages: list

# Tools
def add_numbers(a: int, b: int):
    return {"result": a + b}

TOOLS = {"add_numbers": add_numbers}

# LLM Node
def llm_node(state: AgentState):
    response = client.chat.completions.create(
        model="qwen3:1.7b",
        messages=state["messages"],
        tools=[
            {
                "type": "function",
                "function": {"name": "add_numbers", "parameters": {"type": "object","properties": {}}}
            }
        ],
        tool_choice="auto"
    )

    msg = response.choices[0].message
    return {"messages": state["messages"] + [msg]}

# Tool Node
def tool_node(state: AgentState):
    msg = state["messages"][-1]

    if not msg.tool_calls:
        return END

    tool_call = msg.tool_calls[0]
    args = json.loads(tool_call.function.arguments)

    result = add_numbers(**args)

    return {
        "messages": state["messages"] + [{
            "role": "tool",
            "content": json.dumps(result)
        }]
    }

# Graph
graph = StateGraph(AgentState)
graph.add_node("llm", llm_node)
graph.add_node("tool", tool_node)

graph.set_entry_point("llm")
graph.add_edge("llm", "tool")
graph.add_edge("tool", "llm")

agent = graph.compile()

# Run
def run_agent(query):
    state = {"messages": [{"role": "user", "content": query}]}
    out = agent.invoke(state)
    return out["messages"][-1]["content"]

print(run_agent("Add 5 and 10"))


## 🔧 Advanced Tool Usage

- Multiple tools
- JSON argument parsing
- Function calling
- Structured outputs


## ✅ Next Steps: Extending This Notebook

You can extend this tutorial into the following practical directions:

- Multi-agent systems
- RAG (Retrieval-Augmented Generation)
- GUI apps (Gradio)
- Autonomous workflows (planner → executor loops)

Below are compact, runnable examples and pointers for each topic.


In [ ]:
# Multi-agent systems (simple example with LangGraph)
# Two agents take turns: a 'planner' and an 'executor'.
try:
    from openai import OpenAI
    from langgraph.graph import StateGraph, END
except Exception as e:
    print('Install dependencies: pip install openai langgraph')

# NOTE: This is a minimal illustrative example.
def planner_node(state):
    # planner decides on a simple numeric task
    user_msg = state['messages'][-1]['content']
    if 'add' in user_msg.lower():
        # plan: call executor with args
        plan = {'role': 'assistant', 'content': 'CALL_EXECUTOR: add_numbers', 'meta': {'args': {'a': 5, 'b': 10}}}
        return {'messages': state['messages'] + [plan]}
    return END

def executor_node(state):
    # executor performs the requested function
    last = state['messages'][-1]
    if isinstance(last.get('meta'), dict) and last['content'].startswith('CALL_EXECUTOR'):
        args = last['meta'].get('args', {})
        result = {'result': args.get('a',0) + args.get('b',0)}
        return {'messages': state['messages'] + [{'role': 'tool', 'content': str(result)}]}
    return END

# Compose graph with two nodes representing two agents
from langgraph.graph import StateGraph
class AgentState(dict):
    messages: list

graph = StateGraph(AgentState)
graph.add_node('planner', planner_node)
graph.add_node('executor', executor_node)
graph.set_entry_point('planner')
graph.add_edge('planner','executor')
graph.add_edge('executor','planner')
agent = graph.compile()

def run_multi_agent(query):
    state = {'messages':[{'role':'user','content':query}]}
    out = agent.invoke(state)
    return out['messages'][-1]['content']

print('Multi-agent example output: ', run_multi_agent('Please add two numbers for me'))

## RAG (Retrieval-Augmented Generation)

Below is a minimal pattern: embed documents, build an index (FAISS), retrieve, then call the LLM with the retrieved context. Install: `pip install sentence-transformers faiss-cpu openai`.


In [ ]:
# Minimal RAG example (local, small dataset)
try:
    from sentence_transformers import SentenceTransformer
    import faiss
    import numpy as np
except Exception as e:
    print('Install: pip install sentence-transformers faiss-cpu')

# sample docs
docs = [
    'LangGraph is an agent framework for building control loops.',
    'Retrieval augmented generation combines a retriever and a generator.',
    'Gradio is a quick way to build simple web UIs for ML models.'
]

model = SentenceTransformer('all-MiniLM-L6-v2')
embs = model.encode(docs, convert_to_numpy=True)
d = embs.shape[1]
index = faiss.IndexFlatL2(d)
index.add(embs)

def retrieve(query, k=2):
    qv = model.encode([query], convert_to_numpy=True)
    D, I = index.search(qv, k)
    return [docs[i] for i in I[0]]

print('Retrieved:', retrieve('How do I build agents?'))

## GUI apps with Gradio

Quick demo: wrap your `run_agent` or `run_multi_agent` function with Gradio for a simple web UI. Install: `pip install gradio`.


In [ ]:
# Gradio UI example
try:
    import gradio as gr
except Exception as e:
    print('Install: pip install gradio')

def ui_run(query):
    # prefer run_multi_agent if available
    try:
        return run_multi_agent(query)
    except NameError:
        return 'Agent not configured. Run previous cells.'

# To start the UI, uncomment the next two lines:
# demo = gr.Interface(fn=ui_run, inputs='text', outputs='text', title='Agent UI')
# demo.launch()

print('Gradio UI example ready — uncomment launch lines to run locally')

## Autonomous workflows (planner → executor loop)

A short pattern: build a planner node that decomposes a task into sub-tasks, and an executor node that calls tools. You can schedule retries, add memory checkpoints, and persist state between runs for long-running automation.


In [ ]:
# Autonomous workflow sketch (planner creates steps, executor runs them)
def planner_for_workflow(state):
    # decompose a high-level task into steps
    task = state['messages'][-1]['content']
    steps = [
        {'action':'search_docs','args':{'q':task}},
        {'action':'summarize','args':{}}
    ]
    return {'plan': steps}

def executor_for_workflow(plan):
    results = []
    for step in plan:
        if step['action']=='search_docs':
            results.append({'step':step, 'out':'found docs (mock)'} )
        elif step['action']=='summarize':
            results.append({'step':step, 'out':'summary (mock)'})
    return results

# Example run
state = {'messages':[{'role':'user','content':'Explain how to build a RAG agent'}]}
plan = planner_for_workflow(state)['plan']
print('Plan:', plan)
print('Execution results:', executor_for_workflow(plan))